# Notebook 5: Document Loaders & Text Splitters – Ingesting Real Data

**🎯 Goal:** Learn how to load external documents (like PDFs, TXT files, and CSVs) into a format LangChain can use, and then split them into manageable chunks for processing.

## 🧩 Concept Overview

LLMs have a limited **context window**, which is like their short-term memory. You can't just feed a 100-page document to an LLM at once. We need a way to break it down.

- **Document Loaders:** These are tools that fetch data from a source (a file, a website, a database) and load it as a `Document` object. Each `Document` contains page content (the text) and metadata.
- **Text Splitters:** These tools take a long `Document` and split it into smaller, bite-sized chunks. They are smart enough to try and keep related pieces of text together.

## 🖼️ Visual Diagram

The process looks like this:

```
               +-----------------+      +------------------------+      +----------------------+
Your Data File |                 |      |                        |      | Chunk 1              |
(e.g., PDF) => | Document Loader | ===> | Long Document Object   | ===> | Chunk 2 (overlaps 1) |
               |                 |      | (All text)             |      | Chunk 3 (overlaps 2) |
               +-----------------+      +------------------------+      +----------------------+
                                                 ||                      (many small Documents)
                                                 \/                     
                                          +----------------+
                                          |  Text Splitter | 
                                          +----------------+
```

## ⚙️ Setup

We'll need to install packages for loading different file types, like `pypdf` for PDFs and `unstructured` for more complex files.

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## 1. Document Loaders

Let's try loading the different files we have in our `data` directory.

In [ ]:
# 📄 Loading a .txt file
txt_loader = TextLoader('../data/product-info.txt')
txt_document = txt_loader.load()

print("--- TXT Loader ---")
print(f"Content preview: {txt_document[0].page_content[:100]}...")
print(f"Metadata: {txt_document[0].metadata}\n")

# 📈 Loading a .csv file
csv_loader = CSVLoader('../data/movie-plots.csv')
csv_document = csv_loader.load()

print("--- CSV Loader ---")
print(f"Content preview: {csv_document[0].page_content[:100]}...")
print(f"Metadata: {csv_document[0].metadata}\n")

### A Note on Loading PDFs

To load a real PDF, you would use `PyPDFLoader`. However, since we couldn't create a true PDF in our environment, our `annual-report.pdf` is actually a text file. Therefore, we'll load it with `TextLoader`. 

**If this were a real PDF, the code would be:**
```python
from langchain_community.document_loaders import PyPDFLoader
pdf_loader = PyPDFLoader('../data/annual-report.pdf')
pdf_document = pdf_loader.load()
```

In [ ]:
# 🚨 Loading our 'fake' PDF with TextLoader
pdf_loader = TextLoader('../data/annual-report.pdf')
pdf_document = pdf_loader.load()

print("--- PDF (as TXT) Loader ---")
print(f"Number of pages (documents): {len(pdf_document)}") # Will be 1 for TextLoader
print(f"Content preview: {pdf_document[0].page_content[:200]}...")

## 2. Text Splitters

Now that we have our document loaded, it's one giant block of text. We need to split it.

`RecursiveCharacterTextSplitter` is the recommended choice. It tries to split text on a list of separators (like `\n\n`, `\n`, ` `) in order, which helps keep semantically related pieces of text together.

In [ ]:
# 🔪 Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # The maximum size of a chunk (in characters)
    chunk_overlap=50   # The number of characters to overlap between chunks
)

# Split the PDF document
pdf_chunks = text_splitter.split_documents(pdf_document)

print(f"Original document pages: {len(pdf_document)}")
print(f"Number of chunks after splitting: {len(pdf_chunks)}\n")

print("--- First Chunk ---")
print(pdf_chunks[0].page_content)
print("\n--- Second Chunk ---")
print(pdf_chunks[1].page_content)

## 🔧 Hands-On Exercise

**Goal:** Load the `movie-plots.csv` file and split the documents into chunks.

1.  Load `../data/movie-plots.csv` using `CSVLoader`.
2.  Create a `RecursiveCharacterTextSplitter` with a `chunk_size` of 250 and a `chunk_overlap` of 30.
3.  Split the loaded documents.
4.  Print the number of chunks created.

In [ ]:
# --- Your Code Here ---

# 1. Load the CSV
movie_loader = CSVLoader('../data/movie-plots.csv')
movie_docs = movie_loader.load()

# 2. Create the splitter
movie_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30
)

# 3. Split the documents
movie_chunks = movie_splitter.split_documents(movie_docs)

# 4. Print the number of chunks
print(f"Original number of documents: {len(movie_docs)}")
print(f"Number of chunks after splitting: {len(movie_chunks)}")

## 🐞 Bug Bounty

What happens if you set `chunk_overlap=0`? It works, but it can be problematic. A question about information that falls right on the boundary between two chunks might be impossible for the LLM to answer because it won't see the full context.

**Task:** Split the `pdf_document` again, but this time with `chunk_overlap=0`. Notice how the end of chunk 0 and the beginning of chunk 1 have no text in common.

In [ ]:
no_overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0 # The bug!
)

no_overlap_chunks = no_overlap_splitter.split_documents(pdf_document)

print("--- Chunk 0 (No Overlap) ---")
print(no_overlap_chunks[0].page_content)
print("\n--- Chunk 1 (No Overlap) ---")
print(no_overlap_chunks[1].page_content)

## 💡 Pro Tip

How do you pick the right `chunk_size`? It's a trade-off. 
- **Smaller chunks** (e.g., 200-400 characters) are more specific and can lead to more precise answers, but might miss broader context.
- **Larger chunks** (e.g., 800-1000 characters) provide more context but can be noisier and more expensive to process.

A good starting point is often around **500-1000 characters**, with an overlap of **50-100 characters**. Always experiment to see what works best for your specific documents and use case.

## 🏁 Summary

You've now prepared your external data for use with an LLM!

1.  **Load First:** Use `DocumentLoaders` (`TextLoader`, `PyPDFLoader`, `CSVLoader`, etc.) to read data from various sources into a standard `Document` format.
2.  **Then Split:** Use a `TextSplitter` (like `RecursiveCharacterTextSplitter`) to break large documents into smaller, processable chunks.
3.  **Overlap is Key:** Using a `chunk_overlap` is crucial for maintaining context between chunks, ensuring you don't lose information at the boundaries.